# SRΨ-v2.0 Hybrid Model Training

## 🎯 Objective

Train the hybrid architecture combining:
- **SRΨ's S-Operator**: Spatial robustness (0.74x shift growth)
- **Transformer's Attention**: Temporal conservation (0.24x extrapolation)

## 🔬 Key Innovations

1. **Spatial-Temporal Operator Separation**
2. **Physical Loss Function** (Energy + Momentum constraints)
3. **Zero-Shot Extrapolation Target**: T=500 (10x training length)

---

## Step 0: Setup & Clone Repository

In [ ]:
# Clone repository
!git clone https://github.com/Mozilla2004/srpsi-engine-tiny.git
%cd srpsi-engine-tiny

# CRITICAL: Pull latest changes
!git pull

print("✅ Repository cloned and updated")

## Step 1: Install Dependencies

In [ ]:
# Install PyTorch (if not already installed)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install other dependencies
!pip install pyyaml tqdm numpy matplotlib

print("✅ Dependencies installed")

## Step 2: Check GPU Availability

In [ ]:
import torch

print("🔧 Device Check:")
print(f"   CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"\n🎯 Using device: {device}")

## Step 3: Test SRΨ-v2.0 Model Creation

In [ ]:
import sys
sys.path.append('/content/srpsi-engine-tiny')

from src.models.srpsi_v2_hybrid import create_srpsi_v2_model
import torch

# Create config
cfg = {
    'task': {'tin': 16, 'nx': 128, 'tout': 32},
    'model': {
        'hidden_dim': 128,
        'depth': 4,
        'kernel_size': 5,
        'dropout': 0.1
    }
}

print("🏗️  Creating SRΨ-v2.0 Hybrid Model...")
model = create_srpsi_v2_model(cfg, device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model created with {num_params:,} parameters")

# Test forward pass
batch_size = 4
x = torch.randn(batch_size, 128, 16).to(device)

print(f"\n🧪 Testing forward pass...")
print(f"   Input shape: {x.shape}")

with torch.no_grad():
    output = model(x)

print(f"   Output shape: {output.shape}")
print(f"✅ Forward pass successful!")

## Step 4: Test Physical Loss Function

In [ ]:
from src.training.physical_loss import PhysicalLoss

print("🔧 Testing Physical Loss Function\n")

# Create loss function
loss_fn = PhysicalLoss(
    lambda_energy=0.1,
    lambda_momentum=0.1
)

# Create dummy data
pred = torch.randn(4, 128, 32).to(device)
target = torch.randn(4, 128, 32).to(device)

print(f"Prediction shape: {pred.shape}")
print(f"Target shape: {target.shape}\n")

# Compute loss
total_loss, loss_dict = loss_fn(pred, target, epoch=5)

print("📊 Loss Breakdown (epoch=5):")
for key, value in loss_dict.items():
    print(f"   {key}: {value:.6f}")

print(f"\n✅ Total loss: {total_loss:.6f}")

## Step 5: Start Training

In [ ]:
print("\n" + "="*70)
print(" " * 15 + "STARTING SRΨ-v2.0 TRAINING")
print("="*70)

print("\n🎯 Target Metrics:")
print("   Extrapolation Ratio: < 0.3 (beat Transformer's 0.24x)")
print("   Shift Growth: < 0.8 (beat SRΨ's 0.74x)")
print("   Energy Drift: < 0.3 (beat Transformer's 0.24)")
print("   Momentum Drift: < 2.0 (beat Transformer's 1.18)")

print("\n⏱️  Estimated time: 30-40 minutes on A100\n")

# Start training
!python train_v2_hybrid.py --config config/burgers_v2.yaml --device cuda

## Step 6: Analyze Training Results

In [ ]:
# Load checkpoint
import torch
from pathlib import Path

checkpoint_dir = Path('checkpoints/burgers_v2')
best_checkpoint = checkpoint_dir / 'checkpoint_best.pt'

if best_checkpoint.exists():
    checkpoint = torch.load(best_checkpoint)
    
    print("🏆 Best Model Results:")
    print(f"   Epoch: {checkpoint['epoch']}")
    print(f"   Val Loss: {checkpoint['val_loss']:.4f}")
    print(f"\n   Metrics:")
    for key, value in checkpoint['metrics'].items():
        print(f"   {key}: {value:.4f}")
else:
    print("❌ Checkpoint not found. Training may have failed.")

## Step 7: Zero-Shot Extrapolation Test (T=500)

In [ ]:
# This will be implemented after training completes
print("🚀 Ultimate Challenge: T=500 Extrapolation")
print("   (10x training length)")
print("\n⏳ Coming soon after training completes...")

## Step 8: Download Results

In [ ]:
from google.colab import files

# Download best checkpoint
checkpoint_path = 'checkpoints/burgers_v2/checkpoint_best.pt'
files.download(checkpoint_path)

print("✅ Checkpoint downloaded!")